### Import dependencies

In [ ]:
import json
import gzip
import pandas as pd
import requests

Working with downloaded data

In [33]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as f:
    first_line = json.loads(f.readline())

In [36]:
first_line

{'main_category': 'All Electronics',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'average_rating': 3.5,
 'rating_number': 6,
 'features': [],
 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.

Filter out only recent items (appeared in stock in 2022 or 2023)

In [52]:
def filter_by_date(data: dict) -> dict:
    filter = False
    if int(data["details"]["Date First Available"][-4:]) >= 2022:
        filter = True
    return filter

In [47]:
test_data = {'main_category': 'All Electronics',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'average_rating': 3.5,
 'rating_number': 6,
 'features': [],
 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': None}],
 'videos': [],
 'store': 'Fat Shark',
 'categories': ['Electronics', 'Television & Video', 'Video Glasses'],
 'details': {'Date First Available': 'August 2, 2014',
  'Manufacturer': 'Fatshark'},
 'parent_asin': 'B00MCW7G9M',
 'bought_together': None}

In [61]:
filter_by_date(test_data)

False

Filter data and store in different files

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as f:
    with open ("../../data/meta_Electronics_2022_2023.jsonl", "wt") as f_outPut:
        with open ("../../data/meta_Electronics_before_2022.jsonl", "wt") as f_outPut_before:
            i=0
            for line in f:
                data = json.loads(line.strip())
                try:
                    is_recent = filter_by_date(data)
                    if is_recent:
                        json.dump(data, f_outPut)
                        f_outPut.write("\n")
                        f_outPut.flush()
                except:
                    json.dump(data, f_outPut_before)
                    f_outPut_before.write("\n")
                    f_outPut_before.flush()
                i+=1
                if i%1000==0:
                    print(f"Processed {i} lines")
                    

Split the item in two categories: "has main category", "does not have main category"

In [78]:


def filter_category(data: dict) -> bool:
    filter = False
    if data["main_category"] == None:
        filter = True
    
    return filter
    

In [ ]:
with open("../../data/meta_Electronics_2022_2023.jsonl", "rt") as f:
    with open ("../../data/meta_Electronics_with_category.jsonl", "wt") as f_outPut_category:
        with open ("../../data/meta_Electronics_no_category.jsonl", "wt") as f_outPut_no_category:
            i=0
            for line in f:
                data = json.loads(line.strip())
                
                is_recent = filter_category(data)
                if not is_recent:
                    json.dump(data, f_outPut_category)
                    f_outPut_category.write("\n")
                    f_outPut_category.flush()
                else:
                    json.dump(data, f_outPut_no_category)
                    f_outPut_no_category.write("\n")
                    f_outPut_no_category.flush()
                i+=1
                if i%1000==0:
                    print(f"Processed {i} lines")

Explore distribution by categories

In [4]:
import pandas as pd
df = pd.read_json("../../data/meta_Electronics_with_category.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
df["main_category"].value_counts().plot(kind = "bar")

Filter out items that have at least 100 ratings (df is dataframe)

In [5]:
df_ratings_100 = df[df["rating_number"]>=100]
len(df_ratings_100)

17290

In [ ]:
df_ratings_100["main_category"].value_counts().plot(kind ="bar")

Explore distribution of ratings

In [ ]:
df_ratings_100["average_rating"].plot(kind = "hist", bins=50, range=(0,5))

Sample 1000

In [6]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

In [30]:
len(df_sample_1000)

1000

In [ ]:
df_sample_1000["average_rating"].plot(kind = "hist", bins=50, range=(0,5))

In [ ]:
df_sample_1000["price"].plot(kind = "hist", bins=100, range=(0,500))


In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind = "bar")

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

In [40]:
df_ratings_100.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", orient='records', lines=True)
df_sample_1000.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient='records', lines=True)

Extracting ratings from electronics file that match sampled data

In [42]:
df_ratings_100=pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", lines=True)
df_sample_1000= pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [1]:
import gzip
with gzip.open("../../data/Electronics.jsonl.gz", "rt") as f:
    with open("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", "wt") as f_output:
        id_list = set(df_sample_1000['AssertionErrorparent_asin'].values)
        i=0
        for line in f:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, f_outPut)
                f_outPut.write('\n')
                f_outPut.flush()
            i+=1
            if i%10000 == 0:
                print(f"Processed {i} lines")

NameError: name 'df_sample_1000' is not defined